# Demo - klasyfikacja ras psow (Stanford Dogs)

Notebook laduje wagi zapisane przez `run.py` (bez treningu).
Przed uruchomieniem: `python run.py`, zeby powstal folder `wagi/` (pliki `.pt` oraz `id_to_class.json`).

In [1]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import transforms

from models import build_resnet18, build_resnet50, build_mobilenet
from przygotowanie_danych import ladna_nazwa, przygotuj_dane, NORM_MEAN, NORM_STD

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


## 1. Konfiguracja i mapa klas

In [3]:
MODEL_NAME = "resnet50"      # resnet18 / resnet50 / mobilenet
WAGI_DIR = Path("wagi")

with open(WAGI_DIR / "id_to_class.json", encoding="utf-8") as f:
    id_to_class = {int(k): v for k, v in json.load(f).items()}
num_classes = len(id_to_class)
print("liczba ras:", num_classes)

liczba ras: 120


## 2. Model z zapisanych wag

In [4]:
BUILDERY = {
    "resnet18": build_resnet18,
    "resnet50": build_resnet50,
    "mobilenet": build_mobilenet,
}

def zaladuj(nazwa):
    model = BUILDERY[nazwa](num_classes)
    stan = torch.load(WAGI_DIR / (nazwa + ".pt"), map_location="cpu")
    model.load_state_dict(stan)
    return model.to(device).eval()

model = zaladuj(MODEL_NAME)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/irminasciezor/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 76.9MB/s]


FileNotFoundError: [Errno 2] No such file or directory: 'wagi/resnet50.pt'

## 3. Predykcja

Uzywamy **tego samego** preprocessingu co przy walidacji i tescie
(`Resize(256)` + `CenterCrop(224)`), zeby wyniki byly zgodne z raportem.

In [ ]:
eval_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

def przewidz(sciezka, model=None, k=3):
    model = model or globals()["model"]
    img = Image.open(sciezka).convert("RGB")
    x = eval_tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(x), dim=1)[0]
    p, idx = prob.topk(k)
    return img, [(ladna_nazwa(id_to_class[i.item()]), v.item()) for i, v in zip(idx, p)]

def pokaz(sciezka, prawda=None, model=None):
    img, top = przewidz(sciezka, model)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
    a1.imshow(img)
    a1.axis("off")
    a1.set_title("prawda: " + prawda if prawda else "zdjecie")
    a2.barh([n for n, _ in top][::-1], [v for _, v in top][::-1])
    a2.set_xlim(0, 1)
    a2.set_title("top-3 predykcje")
    plt.tight_layout()
    plt.show()

## 4. Przyklady ze zbioru testowego

Podzial jest deterministyczny (ten sam `SEED`), wiec `test_loader` to dokladnie te zdjecia,
na ktorych liczono wyniki.

In [ ]:
_, _, test_loader, _, _, _ = przygotuj_dane()
test_samples = test_loader.dataset.samples

# 6 losowych zdjec testowych (powtarzalnie dzieki random.seed)
random.seed(0)
for sciezka, label in random.sample(test_samples, 6):
    pokaz(sciezka, prawda=ladna_nazwa(id_to_class[label]))

## 6. Porownanie modeli na jednym zdjeciu

In [ ]:
sciezka, label = test_samples[0]
print("prawda:", ladna_nazwa(id_to_class[label]))
for nazwa in ["resnet18", "resnet50", "mobilenet"]:
    if (WAGI_DIR / (nazwa + ".pt")).exists():
        _, top = przewidz(sciezka, zaladuj(nazwa))
        print(nazwa.ljust(10), [(n, round(v, 2)) for n, v in top])